# Agent-Based SIR Model
## Lesson 3 · Section 2

In this notebook you will build a simple **agent-based epidemic model** step by step.

We will:
1. Define the health states of agents
2. Implement movement on a square grid
3. Test infection and recovery rules on small examples
4. Simulate a full population over time
5. Visualise both spatial spread and SIR time series
6. Explore how randomness changes the outcome
7. End with suggestions for independent work

The notebook is intentionally split into small, testable parts so you can verify each rule before using it in a full simulation.


## 0 · Imports
We use NumPy, Matplotlib, and Python's random module.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('Imports loaded successfully ✓')


---
## 1 · Agent states
Each agent represents one individual. Each individual can be in one of three states:

- `1` = susceptible
- `2` = infected
- `3` = recovered


In [ ]:
STATE_SUSCEPTIBLE = 1
STATE_INFECTED = 2
STATE_RECOVERED = 3

state_names = {
    STATE_SUSCEPTIBLE: 'Susceptible',
    STATE_INFECTED: 'Infected',
    STATE_RECOVERED: 'Recovered',
}

print(state_names)


## 2 · Movement rule
We start with the same simple movement idea as in the lesson code: during each step, an agent moves by one square in one random direction, while staying inside the simulation area.


In [ ]:
def update_position(x, y, area, rng=random):
    direction = rng.randint(1, 4)
    if direction == 1:
        if y < area:
            y += 1
    elif direction == 2:
        if y > 0:
            y -= 1
    elif direction == 3:
        if x > 0:
            x -= 1
    else:
        if x < area:
            x += 1
    return x, y

print(update_position(5, 5, area=10))


### Independent test 1
Test the movement rule at the boundary. The position should remain inside the area.


In [ ]:
random.seed(1)
for _ in range(10):
    x_new, y_new = update_position(0, 0, area=5)
    assert 0 <= x_new <= 5
    assert 0 <= y_new <= 5
print('Test passed ✓ Boundary handling works.')


---
## 3 · Distance-based infection probability
In reality, disease transmission does not require standing on exactly the same spot — proximity is what matters. We model this with a **distance-dependent transmission probability**.

Each infected agent within radius `r_infect` poses an independent transmission risk. The probability from a single infected neighbour at distance $d$ decays exponentially:

$$p_i = p_{\text{infect}} \cdot e^{-d\, /\, r_{\text{infect}}}$$

When multiple infected agents are nearby, the combined probability of being infected by *at least one* of them is:

$$P = 1 - \prod_{i}(1 - p_i)$$

This captures two key effects:
- **Distance matters** — close contacts are far more dangerous than distant ones
- **Multiple exposures compound** — being surrounded by several infected agents is much riskier

In [ ]:
def compute_infection_probability(person_index, positions, states, p_infect, r_infect):
    """Compute infection probability based on proximity to infected agents.
    
    Each infected agent within radius r_infect contributes an independent
    transmission attempt with probability p_infect * exp(-dist / r_infect).
    Returns the combined probability: 1 - prod(1 - p_i).
    """
    if states[person_index] != STATE_SUSCEPTIBLE:
        return 0.0
    
    dx = positions[:, 0].astype(float) - positions[person_index, 0]
    dy = positions[:, 1].astype(float) - positions[person_index, 1]
    distances = np.sqrt(dx**2 + dy**2)
    
    infected = states == STATE_INFECTED
    not_self = np.arange(len(states)) != person_index
    in_range = distances <= r_infect
    
    mask = infected & not_self & in_range
    
    if not np.any(mask):
        return 0.0
    
    probs = p_infect * np.exp(-distances[mask] / r_infect)
    return 1.0 - np.prod(1.0 - probs)

### Independent test 2
Use a tiny hand-made example to verify the distance-based infection probability.

In [ ]:
test_positions = np.array([
    [1, 1],   # person 0: susceptible
    [2, 2],   # person 1: infected, distance ~1.41
    [1, 1],   # person 2: infected, distance 0 (same cell)
    [3, 3],   # person 3: recovered
])

test_states = np.array([
    STATE_SUSCEPTIBLE,
    STATE_INFECTED,
    STATE_INFECTED,
    STATE_RECOVERED,
])

# Person 0 should have non-zero probability (two infected agents nearby)
prob = compute_infection_probability(0, test_positions, test_states, p_infect=0.25, r_infect=3.0)
print(f'Infection probability for person 0: {prob:.4f}')
assert prob > 0
print('Test passed ✓ Distance-based infection probability is positive when infected agents are nearby.')

# Person 3 (recovered) should always get 0
prob_r = compute_infection_probability(3, test_positions, test_states, p_infect=0.25, r_infect=3.0)
assert prob_r == 0.0
print('Test passed ✓ Recovered agent gets zero infection probability.')

---
## 4 · Update the health state of one agent
We now write one function that applies the epidemic rules:

- susceptible + infection probability > 0 → may become infected (random draw against the computed probability)
- infected → may recover with probability `recovery_probability`
- recovered → stays recovered

Note that the function now receives a **pre-computed infection probability** rather than a simple boolean flag.

In [ ]:
def update_health_state(current_state, infection_prob, recovery_probability, rng=random):
    """Compute the next health state for a single agent.
    
    Parameters
    ----------
    current_state : int
        Current state (STATE_SUSCEPTIBLE, STATE_INFECTED, or STATE_RECOVERED).
    infection_prob : float
        Pre-computed probability of becoming infected (from compute_infection_probability).
    recovery_probability : float
        Probability of recovering per step.
    rng : random.Random
        Random number generator.
    """
    if current_state == STATE_SUSCEPTIBLE and infection_prob > 0:
        if rng.random() < infection_prob:
            return STATE_INFECTED
        return STATE_SUSCEPTIBLE

    if current_state == STATE_INFECTED:
        if rng.random() < recovery_probability:
            return STATE_RECOVERED
        return STATE_INFECTED

    return current_state

### Independent test 3
Force infection and recovery with extreme probabilities so the expected result is obvious.

In [ ]:
random.seed(2)
# Susceptible with infection_prob=1.0 must become infected
result_1 = update_health_state(STATE_SUSCEPTIBLE, infection_prob=1.0, recovery_probability=0.0)
# Infected with recovery_prob=1.0 must recover
result_2 = update_health_state(STATE_INFECTED, infection_prob=0.0, recovery_probability=1.0)
# Susceptible with infection_prob=0.0 must stay susceptible
result_3 = update_health_state(STATE_SUSCEPTIBLE, infection_prob=0.0, recovery_probability=0.0)

print('Susceptible exposed with p=1.0 ->', state_names[result_1])
print('Infected with recovery p=1.0 ->', state_names[result_2])
print('Susceptible with no exposure  ->', state_names[result_3])
assert result_1 == STATE_INFECTED
assert result_2 == STATE_RECOVERED
assert result_3 == STATE_SUSCEPTIBLE
print('Test passed ✓ Health-state update works for deterministic cases.')

---
## 5 · Simulate one full time step for all agents
To avoid order effects, we first compute all new health states and only then replace the old array.

For each susceptible agent we call `compute_infection_probability` to get the distance-based probability, then pass it to `update_health_state`.

In [ ]:
def step_agent_sir(positions, states, area, p_infect, recovery_probability, r_infect, rng=random):
    """Advance the simulation by one time step.
    
    1. Compute new health states (using distance-based infection probability).
    2. Move all agents.
    """
    new_states = states.copy()

    for person in range(len(states)):
        prob = compute_infection_probability(person, positions, states, p_infect, r_infect)
        new_states[person] = update_health_state(
            states[person], prob, recovery_probability, rng=rng
        )

    new_positions = positions.copy()
    for person in range(len(states)):
        x, y = update_position(int(positions[person, 0]), int(positions[person, 1]), area, rng=rng)
        new_positions[person] = [x, y]

    return new_positions, new_states

### Independent test 4
Create a tiny example where one susceptible agent is right next to an infected agent — with `p_infect=1.0` it must become infected.

In [ ]:
random.seed(3)
small_positions = np.array([
    [1, 1],   # susceptible
    [1, 2],   # infected, distance = 1
    [4, 4],   # recovered
])
small_states = np.array([
    STATE_SUSCEPTIBLE,
    STATE_INFECTED,
    STATE_RECOVERED,
])

_, updated_states = step_agent_sir(
    small_positions,
    small_states,
    area=5,
    p_infect=1.0,
    recovery_probability=0.0,
    r_infect=3.0,
)

print('Updated states:', [state_names[s] for s in updated_states])
assert updated_states[0] == STATE_INFECTED
print('Test passed ✓ Nearby infection spreads as expected.')

---
## 6 · Full agent-based simulation
Now we combine all previous pieces into one complete model. The new parameter `r_infect` controls the **infection radius** — the maximum distance at which transmission can occur.

In [ ]:
def simulate_agent_sir(
    area=40,
    population=200,
    initially_infected=5,
    steps=150,
    p_infect=0.25,
    recovery_probability=0.03,
    r_infect=3.0,
    seed=1,
):
    """Run a full agent-based SIR simulation with distance-based transmission.
    
    Parameters
    ----------
    r_infect : float
        Maximum distance at which transmission can occur. Probability decays
        exponentially with distance within this radius.
    """
    rng = random.Random(seed)
    positions = np.array(
        [[rng.randint(0, area), rng.randint(0, area)] for _ in range(population)],
        dtype=int,
    )
    states = np.full(population, STATE_SUSCEPTIBLE, dtype=int)
    states[:initially_infected] = STATE_INFECTED

    susceptible_count = np.zeros(steps, dtype=int)
    infected_count = np.zeros(steps, dtype=int)
    recovered_count = np.zeros(steps, dtype=int)
    saved_positions = []
    saved_states = []

    for step in range(steps):
        saved_positions.append(positions.copy())
        saved_states.append(states.copy())
        susceptible_count[step] = np.sum(states == STATE_SUSCEPTIBLE)
        infected_count[step] = np.sum(states == STATE_INFECTED)
        recovered_count[step] = np.sum(states == STATE_RECOVERED)
        positions, states = step_agent_sir(
            positions,
            states,
            area=area,
            p_infect=p_infect,
            recovery_probability=recovery_probability,
            r_infect=r_infect,
            rng=rng,
        )

    return {
        'positions': saved_positions,
        'states': saved_states,
        'susceptible': susceptible_count,
        'infected': infected_count,
        'recovered': recovered_count,
        'final_states': states,
        'area': area,
    }

result = simulate_agent_sir()
print('Simulation finished ✓')
print('Peak infected:', result['infected'].max())

### Independent test 5
The counts of susceptible, infected, and recovered agents should always sum to the total population.


In [ ]:
population = result['susceptible'][0] + result['infected'][0] + result['recovered'][0]
total = result['susceptible'] + result['infected'] + result['recovered']
print('First five totals:', total[:5])
assert np.all(total == population)
print('Test passed ✓ Agent counts are conserved.')


---
## 7 · Visualise one run
We show:
- one spatial snapshot of the agents
- the time series of `S`, `I`, and `R`


In [ ]:
def plot_snapshot_and_curves(simulation_result, snapshot_step=30):
    positions = simulation_result['positions'][snapshot_step]
    states = simulation_result['states'][snapshot_step]
    S = simulation_result['susceptible']
    I = simulation_result['infected']
    R = simulation_result['recovered']
    area = simulation_result['area']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(positions[states == STATE_SUSCEPTIBLE, 0], positions[states == STATE_SUSCEPTIBLE, 1], c='green', s=20, label='S')
    axes[0].scatter(positions[states == STATE_INFECTED, 0], positions[states == STATE_INFECTED, 1], c='red', s=20, label='I')
    axes[0].scatter(positions[states == STATE_RECOVERED, 0], positions[states == STATE_RECOVERED, 1], c='blue', s=20, label='R')
    axes[0].set_xlim(0, area)
    axes[0].set_ylim(0, area)
    axes[0].set_aspect('equal')
    axes[0].set_title(f'Spatial snapshot at step {snapshot_step}')
    axes[0].legend()

    axes[1].plot(S, color='green', linewidth=2, label='S')
    axes[1].plot(I, color='red', linewidth=2, label='I')
    axes[1].plot(R, color='blue', linewidth=2, label='R')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Number of agents')
    axes[1].set_title('Agent-based SIR time series')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_snapshot_and_curves(result, snapshot_step=30)

---
## 8b · Animated Epidemic Spread

The snapshots above show discrete moments, but an animation reveals the full spatial dynamics — how agents wander, how infection clusters grow and merge, and how the recovered barrier gradually forms.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

anim_result = simulate_agent_sir(seed=42, steps=150)
area = anim_result['area']

fig_anim, ax_anim = plt.subplots(figsize=(6, 6))
ax_anim.set_xlim(0, area)
ax_anim.set_ylim(0, area)
ax_anim.set_aspect('equal')
ax_anim.set_xticks([])
ax_anim.set_yticks([])

scat_s = ax_anim.scatter([], [], c='green', s=12, alpha=0.6, label='S')
scat_i = ax_anim.scatter([], [], c='red', s=18, zorder=5, label='I')
scat_r = ax_anim.scatter([], [], c='blue', s=12, alpha=0.6, label='R')
ax_anim.legend(loc='upper right', fontsize=9)
title = ax_anim.set_title('')

def animate(frame):
    pos = anim_result['positions'][frame]
    st = anim_result['states'][frame]
    for state_val, scat in [(STATE_SUSCEPTIBLE, scat_s),
                             (STATE_INFECTED, scat_i),
                             (STATE_RECOVERED, scat_r)]:
        mask = st == state_val
        scat.set_offsets(pos[mask])
    n_i = int(np.sum(st == STATE_INFECTED))
    title.set_text(f'Step {frame}, Infected = {n_i}')
    return scat_s, scat_i, scat_r, title

ani = FuncAnimation(fig_anim, animate, frames=len(anim_result['positions']),
                    interval=80, blit=True)
plt.close(fig_anim)
HTML(ani.to_jshtml())

---
## 8c · Effect of Infection Radius

The parameter `r_infect` controls how far transmission can reach. A larger radius means each infected agent threatens more neighbours per step, dramatically accelerating spread. A very small radius approximates the original same-cell model.

Let's compare epidemic curves for several values of `r_infect`.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Vary infection radius
for r in [1.0, 2.0, 3.0, 5.0]:
    run = simulate_agent_sir(r_infect=r, seed=42)
    ax1.plot(run['infected'], lw=2, label=f'r_infect = {r}')
ax1.set_xlabel('Time step')
ax1.set_ylabel('Infected agents')
ax1.set_title('Effect of Infection Radius', fontweight='bold')
ax1.legend(fontsize=9)

# Vary infection probability
for p in [0.05, 0.15, 0.25, 0.50]:
    run = simulate_agent_sir(p_infect=p, seed=42)
    ax2.plot(run['infected'], lw=2, label=f'p_infect = {p}')
ax2.set_xlabel('Time step')
ax2.set_ylabel('Infected agents')
ax2.set_title('Effect of Infection Probability', fontweight='bold')
ax2.legend(fontsize=9)

# Vary recovery probability
for p_rec in [0.01, 0.03, 0.05, 0.10]:
    run = simulate_agent_sir(recovery_probability=p_rec, seed=42)
    ax3.plot(run['infected'], lw=2, label=f'p_recover = {p_rec}')
ax3.set_xlabel('Time step')
ax3.set_ylabel('Infected agents')
ax3.set_title('Effect of Recovery Probability', fontweight='bold')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Larger infection radius → faster spread, similar to increasing p_infect.')
print('Higher recovery probability → shorter outbreaks with lower peak.')

---
## 8d · Vaccination Experiment

Let's pre-vaccinate some agents (set them to recovered state at the start) and see how this affects the epidemic. This uses the same distance-based transmission model.

In [ ]:
def simulate_abm_sir_vaccinated(vaccinated_fraction=0.0, **kwargs):
    """Run the ABM with a fraction of agents pre-vaccinated (state = recovered)."""
    seed = kwargs.get('seed', 1)
    population = kwargs.get('population', 200)
    area = kwargs.get('area', 40)
    initially_infected = kwargs.get('initially_infected', 5)
    steps = kwargs.get('steps', 150)
    p_infect = kwargs.get('p_infect', 0.25)
    recovery_probability = kwargs.get('recovery_probability', 0.03)
    r_infect = kwargs.get('r_infect', 3.0)
    
    rng = random.Random(seed)
    positions = np.array([[rng.randint(0, area), rng.randint(0, area)]
                          for _ in range(population)], dtype=int)
    states = np.full(population, STATE_SUSCEPTIBLE, dtype=int)
    states[:initially_infected] = STATE_INFECTED
    
    # Vaccinate a fraction of the susceptible population
    n_vacc = int(vaccinated_fraction * (population - initially_infected))
    candidates = list(range(initially_infected, population))
    rng.shuffle(candidates)
    for idx in candidates[:n_vacc]:
        states[idx] = STATE_RECOVERED
    
    s_count = np.zeros(steps, dtype=int)
    i_count = np.zeros(steps, dtype=int)
    r_count = np.zeros(steps, dtype=int)
    
    for step in range(steps):
        s_count[step] = np.sum(states == STATE_SUSCEPTIBLE)
        i_count[step] = np.sum(states == STATE_INFECTED)
        r_count[step] = np.sum(states == STATE_RECOVERED)
        
        new_states = states.copy()
        for p in range(population):
            prob = compute_infection_probability(p, positions, states, p_infect, r_infect)
            new_states[p] = update_health_state(
                states[p], prob, recovery_probability, rng=rng)
        states = new_states
        
        for p in range(population):
            x, y = update_position(int(positions[p, 0]), int(positions[p, 1]), area, rng=rng)
            positions[p] = [x, y]
    
    return {'susceptible': s_count, 'infected': i_count, 'recovered': r_count}

# Run with different vaccination levels
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 5))

for vf, c in zip([0.0, 0.2, 0.4, 0.6, 0.8], colors):
    run = simulate_abm_sir_vaccinated(vaccinated_fraction=vf, seed=42, steps=150)
    ax.plot(run['infected'], color=c, lw=2, label=f'{int(vf*100)}% vaccinated')

ax.set_xlabel('Time step')
ax.set_ylabel('Infected agents')
ax.set_title('Effect of Vaccination Coverage on Epidemic', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('Higher vaccination coverage flattens and suppresses the epidemic curve.')

---
## 9 · Explore randomness
Unlike the equation-based model, the agent-based model is stochastic. Two runs with the same parameters can produce different outcomes.

In [ ]:
seeds = [1, 2, 3, 4]
fig, ax = plt.subplots(figsize=(10, 5))

for seed in seeds:
    run = simulate_agent_sir(seed=seed, steps=120, population=200, initially_infected=5)
    ax.plot(run['infected'], linewidth=2, label=f'seed={seed}')

ax.set_title('Different runs of the same agent-based model')
ax.set_xlabel('Step')
ax.set_ylabel('Infected agents')
ax.legend()
plt.tight_layout()
plt.show()


## 10 · What to try next
Suggested experiments for your own exploration:

- Increase `p_infect` and compare the peak number of infected agents
- Increase `recovery_probability` and see whether the epidemic dies out faster
- Change the `area` while keeping the population fixed and observe the effect of crowding
- Change `r_infect` — how does a very small radius (≈ 0.5) compare to a large one (≈ 8)?
- Change the number of initially infected agents

### Questions to answer
1. Which parameter changes the spread most strongly?
2. Why do different seeds produce different curves?
3. How does `r_infect` interact with `p_infect`? Can a large radius with low probability produce similar curves to a small radius with high probability?
4. In what ways is this model more realistic than the equation-based SIR model?
5. In what ways is it still simplified?

---
## 11 · Independent work and mini-project ideas
Choose one or more of these projects.

### Project A · Add an incubation state
Turn the model into an agent-based SEIR model by adding an exposed state.

### Project B · Add social distancing
Reduce movement or reduce the infection radius after the infected count crosses a threshold.

### Project C · Heterogeneous agents
Give different agents different movement speeds, different recovery probabilities, or different susceptibilities.

### Project D · Compare ABM with the equation-based SIR model
Run both models with similar parameters and compare the epidemic curves.

### Project E · Quarantine zone
When an agent is detected as infected, restrict its movement to a small area. Compare the epidemic with and without quarantine.

### Final reflection
After finishing, describe:
- which features of real epidemics are naturally captured by agents,
- which assumptions are still unrealistic,
- what data you would need to make the model more realistic.